# Scraping Zara.com
Collecte des données produits via Selenium (Firefox headless) + regex sur le HTML rendu.

**Variables collectées :** nom, url, categorie, couleur, matiere, collection, pays, prix, promo, reference

## 0. Installation des dépendances

In [ ]:
# À lancer une seule fois si les packages ne sont pas installés
# !pip install selenium webdriver-manager beautifulsoup4 pandas

## 1. Imports & Configuration

In [2]:
from selenium import webdriver
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.common.by import By
from selenium.common.exceptions import WebDriverException
from webdriver_manager.firefox import GeckoDriverManager
import re
import time
import pandas as pd
from bs4 import BeautifulSoup #Pour extraire et manipuler des données HTML


In [3]:
# Colonnes du DataFrame final
VARIABLES = ["nom", "url", "couleur", "matiere", "prix"]

In [4]:
# URLs de départ — pages liste de produits par collection
# (changer l'URL pour cibler une autre catégorie ou promotion)
URLS_LISTES = {
    "femme":   "https://www.zara.com/fr/fr/femme-nouveau-l1180.html",
    "homme":   "https://www.zara.com/fr/fr/homme-nouveau-l711.html",
    "fille": "https://www.zara.com/fr/fr/enfants-fille-nouveau-l391.html",
    "garçon": "https://www.zara.com/fr/fr/enfants-garcon-nouveau-l228.html"
}

PAYS = "France"

## 2. Lancement du navigateur (Firefox headless)

In [5]:
# headless=True = pas de fenêtre visible (recommandé pour du scraping en batch)
# Mettre headless=False si tu veux voir le navigateur s'ouvrir
HEADLESS = True

options = Options()
if HEADLESS:
    options.add_argument("--headless")

driver = webdriver.Firefox(
    service=Service(GeckoDriverManager().install()),
    options=options
)
print("Navigateur lancé.")

Navigateur lancé.


## 3. Extraction des URLs produits depuis une page liste

In [6]:
def get_product_urls(driver, listing_url, pause=8):
    """
    Charge une page liste Zara et extrait toutes les URLs produit.
    Les URLs produit Zara contiennent '-p' suivi de chiffres.
    """
    driver.get(listing_url)
    time.sleep(pause)

    # Scroll pour charger les produits en lazy-loading
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight / 2);")
    time.sleep(2)
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)

    links = driver.find_elements(By.TAG_NAME, "a")
    urls = set()
    for link in links:
        href = link.get_attribute("href")
        if href and "zara.com/fr/fr/" in href and re.search(r'-p\d+\.html', href):
            # Nettoyer les paramètres d'URL pour garder l'URL canonique
            clean = re.sub(r'\?.*$', '', href)
            urls.add(clean)

    return list(urls)

# Test sur femme
urls_femme = get_product_urls(driver, URLS_LISTES["femme"])
print(f"Femme : {len(urls_femme)} URLs trouvées")
urls_femme[:3]

Femme : 33 URLs trouvées


['https://www.zara.com/fr/fr/robe-midi-fluide-p04772370.html',
 'https://www.zara.com/fr/fr/sandales-a-brides-et-decor-metallique-p12627710.html',
 'https://www.zara.com/fr/fr/pull-asymetrique-en-maille-p09598016.html']

## 5. Fonction de scraping d'un produit

In [8]:
def scrape_produit(driver, url, pause=6):
    """
    Scrape les informations d'une page produit Zara.
    
    Retourne un dict avec :
      nom, url, categorie, couleur, matiere, pays, prix, promo, reference
    """
    try:
        driver.get(url)
        time.sleep(pause)
        html = driver.page_source
    except WebDriverException as e:
        print(f"  [ERREUR WebDriver] {url} : {e}")
        return None
    soup = BeautifulSoup(html, "html.parser")

    # ── NOM ────────────────────────────────────────────────────────────────
    match = re.search(r'"name":"([^"]{3,100})"', html)
    nom = match.group(1) if match else None

    # ── COULEUR ───────────────────────────────────────────────────────────

    color_tag = soup.select_one(
        '[data-qa-qualifier="product-detail-info-color"]'
    )

    if color_tag:
        button = color_tag.find("button")
        if button:
            button.extract()

        couleur = color_tag.get_text(" ", strip=True).strip('"').strip()
    else:
        couleur = None

    # ── MATIÈRE PRINCIPALE ────────────────────────────────────────────────
    match = re.search(
        r'"material":"([^"]+)","percentage":"?([0-9]+)',
        html,
        flags=re.IGNORECASE
    )
    if match:
        matiere_principale = f"{match.group(2)}% {match.group(1)}"
    else:
        matiere_principale = None

    # ── PRIX ──────────────────────────────────────────────────────────────
    price_tag = soup.select_one(".money-amount__main")
    if price_tag:
        prix = price_tag.get_text(" ", strip=True)
    else:
        prix = None

    return {
        "nom":        nom,
        "url":        url,
        "couleur":    couleur,
        "matiere":    matiere_principale,
        "prix":       prix,
    }



In [33]:
# ── TEST sur une URL ─────────────────────────────────────────────────────
if urls_femme:
    test = scrape_produit(driver, urls_femme[0])
    print(test)

{'nom': 'JUPE ASYMÉTRIQUE FLUIDE', 'url': 'https://www.zara.com/fr/fr/jupe-asymetrique-fluide-p01165021.html', 'couleur': 'Sable |', 'matiere': '76% polyester', 'prix': '29,95 EUR'}


## 6. Scraping complet — toutes collections

In [9]:
def scrape_collection(driver, listing_url, pause=3, max=100):
    """
    Récupère toutes les URLs d'une page liste puis scrape chaque produit.
    max_produits : limite le nombre de produits (None = tous).
    """
    urls = get_product_urls(driver, listing_url)
    print(f"  {len(urls)} URLs trouvées")
    urls = urls[:max]

    rows = []
    for i, url in enumerate(urls, 1):
        print(f"  [{i}/{len(urls)}] {url}")
        row = scrape_produit(driver, url,pause=pause)
        if row:
            rows.append(row)

    df = pd.DataFrame(rows, columns=VARIABLES)
    print(f"  → {len(df)} produits récupérés")
    return df

In [10]:
# ── FEMME ────────────────────────────────────────────────────────────────
# max_produits=20 pour un test rapide ; mettre None pour tout scraper
df_femme = scrape_collection(driver, URLS_LISTES["femme"],max=None)
df_femme.head()

  33 URLs trouvées
  [1/33] https://www.zara.com/fr/fr/robe-midi-fluide-p04772370.html
  [2/33] https://www.zara.com/fr/fr/sandales-a-brides-et-decor-metallique-p12627710.html
  [3/33] https://www.zara.com/fr/fr/pull-asymetrique-en-maille-p09598016.html
  [4/33] https://www.zara.com/fr/fr/combinaison-courte-en-lyocell-p06929485.html
  [5/33] https://www.zara.com/fr/fr/jean-trf-loose-folded-taille-basse-a-rayures-p08727114.html
  [6/33] https://www.zara.com/fr/fr/robe-fluide-a-encolure-americaine-p02180312.html
  [7/33] https://www.zara.com/fr/fr/veste-croisee-p04387204.html
  [8/33] https://www.zara.com/fr/fr/lot-de-3-bracelets-bois-et-resine-p07243005.html
  [9/33] https://www.zara.com/fr/fr/sandales-a-talons-imprimees-p13808810.html
  [10/33] https://www.zara.com/fr/fr/short-fluide-a-boutons-p06929421.html
  [11/33] https://www.zara.com/fr/fr/t-shirt-texte-brillant-effet-delave-p06050078.html
  [12/33] https://www.zara.com/fr/fr/veste-a-rayures-100-lin-p02775647.html
  [13/33] https:

,nom,url,couleur,matiere,prix
0,ROBE MI-LONGUE FLUIDE,https://www.zara.com/fr/fr/robe-midi-fluide-p0...,Mauve poudré |,88% viscose,"25,99 EUR"
1,SANDALES À BRIDES ET DÉCOR MÉTALLIQUE,https://www.zara.com/fr/fr/sandales-a-brides-e...,Marron |,100% polyuréthane,"39,95 EUR"
2,PNG,https://www.zara.com/fr/fr/pull-asymetrique-en...,None,None,"25,95 EUR"
3,COMBI-SHORT LYOCELL,https://www.zara.com/fr/fr/combinaison-courte-...,Noir |,100% lyocell,"35,95 EUR"
4,JEAN TRF LOOSE FOLDED TAILLE BASSE À RAYURES,https://www.zara.com/fr/fr/jean-trf-loose-fold...,unique |,100% coton,"35,95 EUR"


In [12]:
# ── HOMME ────────────────────────────────────────────────────────────────
df_homme = scrape_collection(driver, URLS_LISTES["homme"])
df_homme.head()

  118 URLs trouvées
  [1/100] https://www.zara.com/fr/fr/lunettes-de-soleil-rectangulaires-p02750301.html
  [2/100] https://www.zara.com/fr/fr/blazer-regular-fit-en-daim-p02521169.html
  [3/100] https://www.zara.com/fr/fr/sac-shopper-en-tissu-p13352720.html
  [4/100] https://www.zara.com/fr/fr/t-shirt-imprime-contraste-a-piece-p06224203.html
  [5/100] https://www.zara.com/fr/fr/pantalon-plisse-de-costume-100-lin-p04504783.html
  [6/100] https://www.zara.com/fr/fr/lunettes-de-soleil--il-de-chat-p02727301.html
  [7/100] https://www.zara.com/fr/fr/lunettes-de-soleil-rectangulaires-p02750300.html
  [8/100] https://www.zara.com/fr/fr/blouson-technique-relaxed-fit-p01437441.html
  [9/100] https://www.zara.com/fr/fr/chemise-a-carreaux-broderie-contraste-p07446424.html
  [10/100] https://www.zara.com/fr/fr/jorts-en-denim-a-ceinture-p01416477.html
  [11/100] https://www.zara.com/fr/fr/mocassins-en-cuir-avec-masque-p12678720.html
  [12/100] https://www.zara.com/fr/fr/sac-bandouliere-en-cuir-p136

,nom,url,couleur,matiere,prix
0,LUNETTES DE SOLEIL RECTANGULAIRES,https://www.zara.com/fr/fr/lunettes-de-soleil-...,Rouge foncé |,100% acétate,"59,95 EUR"
1,BLAZER REGULAR FIT EN DAIM,https://www.zara.com/fr/fr/blazer-regular-fit-...,Sable |,100% cuir de chèvre,"199,00 EUR"
2,SAC SHOPPER EN TISSU,https://www.zara.com/fr/fr/sac-shopper-en-tiss...,Blanc écru |,100% coton,"29,95 EUR"
3,T-SHIRT IMPRIMÉ CONTRASTANT AVEC PIÈCE,https://www.zara.com/fr/fr/t-shirt-imprime-con...,Bleu ciel |,100% coton,"22,95 EUR"
4,PANTALON À PLIS DE COSTUME 100 % LIN,https://www.zara.com/fr/fr/pantalon-plisse-de-...,Blanc |,100% lin,"59,95 EUR"


In [13]:
# ── ENFANTS ──────────────────────────────────────────────────────────────
df_fille = scrape_collection(driver, URLS_LISTES["fille"])
df_fille.head()

  147 URLs trouvées
  [1/100] https://www.zara.com/fr/fr/6-14-ans--bikini-brillant-p01492640.html
  [2/100] https://www.zara.com/fr/fr/ensemble-t-shirt-et-bermuda-a-rayures-p09144607.html
  [3/100] https://www.zara.com/fr/fr/ballerines-sport-barefoot-p12331730.html
  [4/100] https://www.zara.com/fr/fr/pantalon-wide-leg-en-molleton-penn---p09000002.html
  [5/100] https://www.zara.com/fr/fr/short-a-rayures-penn---p06345850.html
  [6/100] https://www.zara.com/fr/fr/top-halter-fantaisie-p03641601.html
  [7/100] https://www.zara.com/fr/fr/lot-de-deux-colliers-avec-fantasie-et-coquillages-p07243585.html
  [8/100] https://www.zara.com/fr/fr/jupe-a-plis-plats-penn---p05987619.html
  [9/100] https://www.zara.com/fr/fr/top-polo-a-rayures-penn---p05987618.html
  [10/100] https://www.zara.com/fr/fr/baskets-de-sport-combinees-p14330730.html
  [11/100] https://www.zara.com/fr/fr/polo-brode-penn---p05871704.html
  [12/100] https://www.zara.com/fr/fr/2-6-ans--maillot-de-bain-fleur-hibiscus-p05644438.h

,nom,url,couleur,matiere,prix
0,MAILLOT DE BAIN DEUX PIÈCES BRILLANT 6-14 ANS,https://www.zara.com/fr/fr/6-14-ans--bikini-br...,Bleu clair |,76% polyamide,"19,95 EUR"
1,ENSEMBLE T-SHIRT ET BERMUDA À RAYURES,https://www.zara.com/fr/fr/ensemble-t-shirt-et...,Bleu |,62% polyester,"17,95 EUR"
2,BALLERINES SPORT BAREFOOT,https://www.zara.com/fr/fr/ballerines-sport-ba...,Blanc écru |,63% polyester,"35,95 EUR"
3,PANTALON EN MOLLETON WIDE LEG PENN ©,https://www.zara.com/fr/fr/pantalon-wide-leg-e...,Vert / Écru |,100% coton,"22,95 EUR"
4,SHORT À RAYURES PENN ©,https://www.zara.com/fr/fr/short-a-rayures-pen...,Bleu clair |,99% coton,"19,95 EUR"


In [14]:
df_garcon = scrape_collection(driver, URLS_LISTES["garçon"])
df_garcon.head()

  167 URLs trouvées
  [1/100] https://www.zara.com/fr/fr/sweat-uni-teint-en-piece-p00935720.html
  [2/100] https://www.zara.com/fr/fr/casquette-en-resille-minecraft---mojang-ab----p00547547.html
  [3/100] https://www.zara.com/fr/fr/6-14-ans--short-de-bain-uni-p05644466.html
  [4/100] https://www.zara.com/fr/fr/t-shirt-sport-a-inscription-p08054697.html
  [5/100] https://www.zara.com/fr/fr/pantalon-wide-leg-en-molleton-penn---p09000002.html
  [6/100] https://www.zara.com/fr/fr/short-a-rayures-penn---p06345850.html
  [7/100] https://www.zara.com/fr/fr/sweat-imprime-p05372630.html
  [8/100] https://www.zara.com/fr/fr/ensemble-t-shirt-et-bermuda-uni-teint-en-piece-p05048707.html
  [9/100] https://www.zara.com/fr/fr/jean-relaxed-ballon-p06186601.html
  [10/100] https://www.zara.com/fr/fr/jupe-a-plis-plats-penn---p05987619.html
  [11/100] https://www.zara.com/fr/fr/top-polo-a-rayures-penn---p05987618.html
  [12/100] https://www.zara.com/fr/fr/t-shirt-imprime-chupa-chups---p05431936.html
  [1

,nom,url,couleur,matiere,prix
0,PNG,https://www.zara.com/fr/fr/sweat-uni-teint-en-...,None,None,"15,99 EUR"
1,CASQUETTE RÉSILLE MINECRAFT © MOJANG AB. ™,https://www.zara.com/fr/fr/casquette-en-resill...,Vert |,100% coton,"9,95 EUR"
2,6-14 ANS / SHORT DE BAIN UNI,https://www.zara.com/fr/fr/6-14-ans--short-de-...,Citron vert |,100% polyamide,"16,95 EUR"
3,T-SHIRT SPORT À INSCRIPTION,https://www.zara.com/fr/fr/t-shirt-sport-a-ins...,Mandarine |,52% polyester,"12,95 EUR"
4,PANTALON EN MOLLETON WIDE LEG PENN ©,https://www.zara.com/fr/fr/pantalon-wide-leg-e...,Vert / Écru |,100% coton,"22,95 EUR"


In [15]:
# Fermer le navigateur proprement
driver.quit()
print("Navigateur fermé.")

Navigateur fermé.


In [16]:
df_femme["collection"]="femme"
df_homme["collection"]="homme"
df_fille["collection"]="fille"
df_garcon["collection"]="garçon"

In [17]:
df_total = pd.concat([df_femme, df_homme, df_fille,df_garcon], ignore_index=True)
df_total["categorie"]=df_total["nom"].str.split().str[0]
df_total.head(10)

,nom,url,couleur,matiere,prix,collection,categorie
0,ROBE MI-LONGUE FLUIDE,https://www.zara.com/fr/fr/robe-midi-fluide-p0...,Mauve poudré |,88% viscose,"25,99 EUR",femme,ROBE
1,SANDALES À BRIDES ET DÉCOR MÉTALLIQUE,https://www.zara.com/fr/fr/sandales-a-brides-e...,Marron |,100% polyuréthane,"39,95 EUR",femme,SANDALES
2,PNG,https://www.zara.com/fr/fr/pull-asymetrique-en...,None,None,"25,95 EUR",femme,PNG
3,COMBI-SHORT LYOCELL,https://www.zara.com/fr/fr/combinaison-courte-...,Noir |,100% lyocell,"35,95 EUR",femme,COMBI-SHORT
4,JEAN TRF LOOSE FOLDED TAILLE BASSE À RAYURES,https://www.zara.com/fr/fr/jean-trf-loose-fold...,unique |,100% coton,"35,95 EUR",femme,JEAN
5,ROBE FLUIDE À ENCOLURE AMÉRICAINE,https://www.zara.com/fr/fr/robe-fluide-a-encol...,Vert huile |,86% viscose,"35,95 EUR",femme,ROBE
6,VESTE CROISÉE,https://www.zara.com/fr/fr/veste-croisee-p0438...,brique foncée |,89% viscose,"32,95 EUR",femme,VESTE
7,LOT DE 3 BRACELETS BOIS ET RÉSINE,https://www.zara.com/fr/fr/lot-de-3-bracelets-...,Multicolore |,60% bois de manguier,"19,95 EUR",femme,LOT
8,SANDALES À TALONS IMPRIMÉES,https://www.zara.com/fr/fr/sandales-a-talons-i...,Léopard |,100% polyester,"32,95 EUR",femme,SANDALES
9,SHORT FLUIDE AVEC BOUTONS,https://www.zara.com/fr/fr/short-fluide-a-bout...,Mauve clair |,83% viscose,"22,95 EUR",femme,SHORT


In [18]:
# Export CSV
df_total.to_csv("zara_data.csv", index=False, encoding="utf-8-sig")
print("Fichier 'zara_data.csv' exporté.")

Fichier 'zara_data.csv' exporté.
